<a href="https://colab.research.google.com/github/dudinha-web/fundamentos-de-ia/blob/main/Exemplo_Aula_07_RAG_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# Métricas para RAG

O framework mais usado para avaliar RAG é o **RAGAS** (RAG Assessment).
Ele mede 3 dimensões principais:

- **Faithfulness** → A resposta é fiel ao contexto recuperado? (sem alucinações)
- **Answer Relevancy** → A resposta responde de facto à pergunta?
- **Context Recall** → O retriever buscou os trechos certos?

In [ ]:
# Instalação das dependências
!pip install -q ragas langchain-community langchain-openai chromadb sentence-transformers datasets

In [ ]:
!pip install -q langchain-groq


In [ ]:
!pip install langchain_huggingface

In [ ]:
#

from datasets import Dataset

dados_avaliacao = [
    {
        "question": "Qual é o objetivo do documento?",
        "answer": "O documento tem como objetivo apresentar o plano anual de metas da empresa.",
        "contexts": [
            "Este documento apresenta o plano anual de metas e KPIs da organização para 2024.",
            "O relatório cobre as áreas de vendas, marketing e operações."
        ],
        "ground_truth": "Apresentar o plano anual de metas da empresa para 2024."
    },
    {
        "question": "Qual foi o crescimento de vendas no Q3?",
        "answer": "O crescimento foi de 15% no terceiro trimestre.",
        "contexts": [
            "No terceiro trimestre, as vendas cresceram 15% em relação ao mesmo período do ano anterior."
        ],
        "ground_truth": "15% de crescimento no Q3."
    },
    {
        "question": "Quem assinou o contrato?",
        "answer": "O CEO João Silva assinou o contrato.",
        "contexts": [
            "O contrato foi assinado em janeiro de 2024 pelo diretor financeiro."
        ],
        "ground_truth": "O diretor financeiro assinou o contrato."
    }
]

dataset = Dataset.from_list(dados_avaliacao)
print(f"Dataset criado com {len(dataset)} amostras")
print(dataset)

In [ ]:
#
import os
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

os.environ["GROQ_API_KEY"] = ""

llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
embeddings_hf = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision

faithfulness.llm = LangchainLLMWrapper(llm_groq)
answer_relevancy.llm = LangchainLLMWrapper(llm_groq)
answer_relevancy.embeddings = LangchainEmbeddingsWrapper(embeddings_hf)
context_recall.llm = LangchainLLMWrapper(llm_groq)
context_precision.llm = LangchainLLMWrapper(llm_groq)

resultado_groq = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_recall, context_precision],
)
print(resultado_groq)

In [ ]:

#resultados em tabela

import pandas as pd
import matplotlib.pyplot as plt

df = resultado_groq.to_pandas()
print(df[['faithfulness', 'answer_relevancy', 'context_recall', 'context_precision']])


metricas = ['faithfulness', 'answer_relevancy', 'context_recall', 'context_precision']
medias = df[metricas].mean()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(metricas, medias, color=['#4CAF50','#2196F3','#FF9800','#9C27B0'])
ax.set_ylim(0, 1)
ax.set_ylabel('Score (0 a 1)')
ax.set_title('Médias')
for bar, val in zip(bars, medias):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('metricas_ragas.png', dpi=150)
plt.show()